# Cleaning and Transformation of merged Dataset

This Jupyter Notebook builds up on the result of *explore_indicator_data.ipynb* where a first data cleaning was done. The 4 original datasets were merged on the smallest (bulk density dataset) to a dataset with all columns of interest combined, as they are needed to prepare SHI calculation.

**! ! ! Run *explore_indicator_data.ipynb* first to receive the result *df_all_coi_merged.csv* in the output folder. Otherwise this Notebook won't work ! ! !**

### CONTENT
1) Data Cleaning
    + reduce to neccessary columns
    + turn empty cells into NaN

2) Data Transformation
    + rename and refill Depth & EC columns
    + check datatypes
    + rename all columns to have units
    + change '< LOD' values according to agreement (random numbers between min and official LOD named in LUCAS documentation)
    + add, calculate and transform all columns
    + calculate quantiles for OC and Clay:SOC ratio
    + use quantiles and predefined bins to determine a score for each indicator in each sample


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 1. Data Cleaning

### Reduction of columns and removal of empty stings ''
(Clean #0)

In [2]:
# load the result result of *explore_indicator_data.ipynb*
df_clean_0 = pd.read_csv("../output/df_all_coi_merged.csv")

# drop unused columns for better overview, not necessary 'LC0_Desc', 'LU1_Desc'
df_clean_0.drop(columns=['LC', 'LU', 'LC1_Desc', 'TH_LAT', 'TH_LONG'], inplace=True)

# replace all missing values with NaN
df_clean_0.replace('', np.nan, inplace=True)

# delete all rows that have NaN now and copy all kept to a new df
df_clean_0 = df_clean_0.dropna().copy()

# save the df
df_clean_0.to_csv("../output/df_clean_0.csv", index=False)

# print to check
print("Dataset Size after deleting NaN:", len(df_clean_0))

# check
df_clean_0.head()


Dataset Size after deleting NaN: 4625


,POINT_ID,BD 0-20,Depth,pH_CaCl2,EC,OC,P,N,K,LC0_Desc,LU1_Desc,Clay,erosion_by_water
1,26762002,1.420,0-20 cm,7.0,12.41,8.6,54,0.7,204.5,Cropland,Agriculture (excluding fallow land and kitchen...,7.0,0.150000
2,27842416,0.955,0-20 cm,3.7,16.56,91.6,14.6,6.9,61.2,Woodland,Forestry,13.0,0.025000
3,26942016,1.089,0-20 cm,3.9,5.89,15,< LOD,1.2,38.2,Woodland,Forestry,13.0,0.175000
4,26962018,1.088,0-20 cm,5.0,12.67,39.8,< LOD,2.4,110.1,Woodland,Forestry,18.0,0.125004
5,30303360,0.943,0-20 cm,4.8,14.18,54.9,40.2,4.8,90.7,Grassland,Agriculture (excluding fallow land and kitchen...,13.0,0.000000


### Cleaning of Depth column
(Clean #1)

In [3]:
# for cell independence load df_clean_0 from csv for depth transformation
df_clean_1 = pd.read_csv("../output/df_clean_0.csv")

# keep only data samples in the dataframe where Depth contains '0-20 cm'
df_clean_1 = df_clean_1[df_clean_1['Depth'] == '0-20 cm']

# make a new depth column and fill it with integer 20, as Depth: '0-20 cm' is a string
df_clean_1["depth_cm"] = 20

# drop the now useless Depth column
df_clean_1.drop(columns=['Depth'], inplace=True)

# save df
df_clean_1.to_csv("../output/df_clean_1.csv", index=False)

print("Dataset Size after deleting wrong Depths:", len(df_clean_1))
df_clean_1.head()

Dataset Size after deleting wrong Depths: 4621


,POINT_ID,BD 0-20,pH_CaCl2,EC,OC,P,N,K,LC0_Desc,LU1_Desc,Clay,erosion_by_water,depth_cm
0,26762002,1.420,7.0,12.41,8.6,54,0.7,204.5,Cropland,Agriculture (excluding fallow land and kitchen...,7.0,0.150000,20
1,27842416,0.955,3.7,16.56,91.6,14.6,6.9,61.2,Woodland,Forestry,13.0,0.025000,20
2,26942016,1.089,3.9,5.89,15,< LOD,1.2,38.2,Woodland,Forestry,13.0,0.175000,20
3,26962018,1.088,5.0,12.67,39.8,< LOD,2.4,110.1,Woodland,Forestry,18.0,0.125004,20
4,30303360,0.943,4.8,14.18,54.9,40.2,4.8,90.7,Grassland,Agriculture (excluding fallow land and kitchen...,13.0,0.000000,20


## 2. Data Transformation

### Handle '< LOD' values
(Transform #0)


Results of exploration:
- OC LOD = 2.0, 0 samples below, '< LOD' string in 5 samples, OC min = 2.2
- P LOD = 10.0, 32 samples below, '< LOD' string in 1711 samples, P min = 0.3
- N LOD = 0.2, 0 samples below, '< LOD' string in 2 samples, N min = 0.2
- K LOD = 10.0, 2 samples below, '< LOD' string in 24 samples, K min = 6.2

+ '< LOD' occur in 4 columns (OC, N, P, K)
+ for following calculations these values need to be numeric
+ approach:
    + export the current dataframe *df_no_empty_strings* as *df_no_empty_strings_lod_orig.csv*, make a working copy of the dataframe *df_no_empty_strings* called *df_transform*
    + assign to each '< LOD' occurence a random number between the minimum value and the official LOD value of that indicator (which was taken from LUCAS documentation)
    + we use the same approach for both cases:
        + for P and K: min value is below the LOD
        + for N and OC: min value is not below their LOD

_The fact that in some cases the measured minimum value is lower than LOD (P, K) is a contradiction, here these samples are kept. This method assumes that values below LOD are uniformly distributed between min and LOD which may not always be true. Uncertainty might be introduced, especially for P with around 1700 '< LOD' occurences. More sophisticated imputation methods would be i.e. maximum likelihood, Bayesian approaches, if precision is very critical._

In [4]:
df_transform_0 = pd.read_csv("../output/df_clean_1.csv")

# create a dictionary for the columns with '< LOD' and their specific min value (see explore_indicator_data.ipynb)
# different approaches are possible: mean of median of all values below official LOD (...)
indicator_lod_dict = {
    'OC': 2.0,
    'N': 0.2,
    'P': 10.0,
    'K': 10.0
}

# Set a seed for reproducibility
np.random.seed(28)

# Loop through each column
for indicator, lod in indicator_lod_dict.items():
    indicator_min = pd.to_numeric(df_transform_0[indicator], errors='coerce').min()
    df_transform_0[indicator] = df_transform_0[indicator].apply(
        lambda x: round(np.random.uniform(indicator_min, lod), 1) if x == '< LOD' else float(x)
    )

# SANITY CHECK - df should be free of NaN, '< LOD', empty strings or negative values
# Check for NaN values in the entire DataFrame
nan_counts = df_transform_0.isna().sum().sum()
print("Number of NaNs in the DataFrame: ", nan_counts)
# Check for empty strings in the entire DataFrame
empty_counts = (df_transform_0 == '').sum().sum()
print("Number of empty strings in the DataFrame: ", empty_counts)
# Count all cells that contain '< LOD' in the entire DataFrame
total_lod = (df_transform_0.astype(str).apply(lambda col: col.str.contains('< LOD'))).sum().sum()
print("Total number of '< LOD' cells in the DataFrame:", total_lod)
# check negative values
neg_values_present = (df_transform_0.select_dtypes(include='number') < 0).any().any()
print("Negative values in DataFrame? ", neg_values_present)

print("Dataset Size:", len(df_transform_0))

# save
df_transform_0.to_csv("../output/df_transform_0.csv", index=False)

# Check the result
df_transform_0.head()

Number of NaNs in the DataFrame:  0
Number of empty strings in the DataFrame:  0
Total number of '< LOD' cells in the DataFrame: 0
Negative values in DataFrame?  False
Dataset Size: 4621


,POINT_ID,BD 0-20,pH_CaCl2,EC,OC,P,N,K,LC0_Desc,LU1_Desc,Clay,erosion_by_water,depth_cm
0,26762002,1.420,7.0,12.41,8.6,54.0,0.7,204.5,Cropland,Agriculture (excluding fallow land and kitchen...,7.0,0.150000,20
1,27842416,0.955,3.7,16.56,91.6,14.6,6.9,61.2,Woodland,Forestry,13.0,0.025000,20
2,26942016,1.089,3.9,5.89,15.0,2.1,1.2,38.2,Woodland,Forestry,13.0,0.175000,20
3,26962018,1.088,5.0,12.67,39.8,8.6,2.4,110.1,Woodland,Forestry,18.0,0.125004,20
4,30303360,0.943,4.8,14.18,54.9,40.2,4.8,90.7,Grassland,Agriculture (excluding fallow land and kitchen...,13.0,0.000000,20


### Check datatypes
The dataframe has by now almost only numeric values, apart from landcover (LC0_Desc) and landuse (LU1_Desc). Pandas automatically applies the correct datatypes. We do not need to transform explicitly, but we check them in a first step.

### Transform EC and Erosion, rename columns (assign units)
(Transform #1)

In [5]:
# for cell independence load df from csv
df_transform_1 = pd.read_csv("../output/df_transform_0.csv")

# check dtypes, only LC and LU should be strings, everything else numeric
print(df_transform_1.dtypes)

# we need EC in deciSiemens per meter, it comes in milliSiemens per meter, so /100 or * 0.01
df_transform_1["EC"] = (df_transform_1["EC"]/100).round(4)

# round erosion_by_water to only 2 decimal places
df_transform_1["erosion_by_water"] = (df_transform_1["erosion_by_water"]).round(2)


# Rename columns, integrate units for clarity
df_transform_1.rename(columns={"BD 0-20": "bulk_density_g/cm3"}, inplace=True)
df_transform_1.rename(columns={"EC": "EC_dS/m"}, inplace=True)
df_transform_1.rename(columns={"OC": "OC_g/kg"}, inplace=True)
df_transform_1.rename(columns={"P": "P_mg/kg"}, inplace=True)
df_transform_1.rename(columns={"N": "N_g/kg"}, inplace=True)
df_transform_1.rename(columns={"K": "K_mg/kg"}, inplace=True)
df_transform_1.rename(columns={"Clay": "Clay_pct"}, inplace=True)
df_transform_1.rename(columns={"erosion_by_water": "erosion_by_water_t/ha/yr"}, inplace=True)
df_transform_1.rename(columns={"LC0_Desc": "Landcover"}, inplace=True)
df_transform_1.rename(columns={"LU1_Desc": "Landuse"}, inplace=True)

# save
df_transform_1.to_csv("../output/df_transform_1.csv", index=False)

# Check the result
df_transform_1.head()

POINT_ID              int64
BD 0-20             float64
pH_CaCl2            float64
EC                  float64
OC                  float64
P                   float64
N                   float64
K                   float64
LC0_Desc                str
LU1_Desc                str
Clay                float64
erosion_by_water    float64
depth_cm              int64
dtype: object


,POINT_ID,bulk_density_g/cm3,pH_CaCl2,EC_dS/m,OC_g/kg,P_mg/kg,N_g/kg,K_mg/kg,Landcover,Landuse,Clay_pct,erosion_by_water_t/ha/yr,depth_cm
0,26762002,1.420,7.0,0.1241,8.6,54.0,0.7,204.5,Cropland,Agriculture (excluding fallow land and kitchen...,7.0,0.15,20
1,27842416,0.955,3.7,0.1656,91.6,14.6,6.9,61.2,Woodland,Forestry,13.0,0.02,20
2,26942016,1.089,3.9,0.0589,15.0,2.1,1.2,38.2,Woodland,Forestry,13.0,0.18,20
3,26962018,1.088,5.0,0.1267,39.8,8.6,2.4,110.1,Woodland,Forestry,18.0,0.13,20
4,30303360,0.943,4.8,0.1418,54.9,40.2,4.8,90.7,Grassland,Agriculture (excluding fallow land and kitchen...,13.0,0.00,20


### Calculation of N, P, K stocks in kg/ha and the Clay:SOC ratio
(Transform #2)

In [6]:
# for cell independence load df from csv
df_transform_2 = pd.read_csv("../output/df_transform_1.csv")

# defining a function to calculate the stock values in kg/ha for N, P and K
# it takes the value in the column (N, P or K) as concentration, bulk_density and depth as itself and unit has to be entered!
# for N it is g/kg and for P and K it is mg/kg
# the returned value is always in kg/ha (N, P or K)

def calc_stock(concentration, bulk_density, depth, unit):
    factors = {
        "mg/kg": 0.1,
        "g/kg": 100
    }
    
    if unit not in factors:
        raise ValueError(f"Unit '{unit}' not recognized. Use 'mg/kg' or 'g/kg'.")

    stock = (concentration * bulk_density * depth * factors[unit]).round(4)
    return stock

# function returns a number that represents the ratio of clay to SOC
# we multiply by 10 as OC comes in g/kg which is 0,1% but we want 1% as Clay also comes in %, it is the same as doing clay / (soc : 10)
# in case soc is 0, we cannot divide by 0, NaN would be the result
def calc_clay_SOC_ratio (clay, soc):
    ratio = np.where(soc == 0, np.nan, (clay / soc * 10).round(4))
    return ratio

# calculate the stocks for each element using the function calc_stock
df_transform_2["P_kg/ha"] = calc_stock(
    df_transform_2["P_mg/kg"],
    df_transform_2["bulk_density_g/cm3"],
    df_transform_2["depth_cm"],
    unit="mg/kg"
)

df_transform_2["N_kg/ha"] = calc_stock(
    df_transform_2["N_g/kg"],
    df_transform_2["bulk_density_g/cm3"],
    df_transform_2["depth_cm"],
    unit="g/kg"
)

df_transform_2["K_kg/ha"] = calc_stock(
    df_transform_2["K_mg/kg"],
    df_transform_2["bulk_density_g/cm3"],
    df_transform_2["depth_cm"],
    unit="mg/kg"
)

df_transform_2["Clay_SOC_ratio"] = calc_clay_SOC_ratio(
    df_transform_2["Clay_pct"],
    df_transform_2["OC_g/kg"]
)

# rounding the new columns to 2 digits
cols_to_round = ["N_kg/ha", "P_kg/ha", "K_kg/ha", "Clay_SOC_ratio"]
df_transform_2[cols_to_round] = df_transform_2[cols_to_round].round(2)

# drop columns not needed anymore for clarity
df_transform_2.drop(columns=["P_mg/kg", "N_g/kg", "K_mg/kg", "Clay_pct", "depth_cm"], inplace=True)

# save
df_transform_2.to_csv("../output/df_transform_2.csv", index=False)

# check
df_transform_2.head()

,POINT_ID,bulk_density_g/cm3,pH_CaCl2,EC_dS/m,OC_g/kg,Landcover,Landuse,erosion_by_water_t/ha/yr,P_kg/ha,N_kg/ha,K_kg/ha,Clay_SOC_ratio
0,26762002,1.420,7.0,0.1241,8.6,Cropland,Agriculture (excluding fallow land and kitchen...,0.15,153.36,1988.0,580.78,8.14
1,27842416,0.955,3.7,0.1656,91.6,Woodland,Forestry,0.02,27.89,13179.0,116.89,1.42
2,26942016,1.089,3.9,0.0589,15.0,Woodland,Forestry,0.18,4.57,2613.6,83.20,8.67
3,26962018,1.088,5.0,0.1267,39.8,Woodland,Forestry,0.13,18.71,5222.4,239.58,4.52
4,30303360,0.943,4.8,0.1418,54.9,Grassland,Agriculture (excluding fallow land and kitchen...,0.00,75.82,9052.8,171.06,2.37


### Quantiles & Bins for indicator score calculation
_In order to assign one score per indicator per sample, a scale or bins are needed._

(Transform #3)

#### Quantiles and score calculation for Clay:SOC ratio and OC

In [7]:
# for cell independence load df from csv
df_transform_3 = pd.read_csv("../output/df_transform_2.csv")

# defining the quantiles
quantiles = [0.25, 0.50, 0.75]
quantile_clay_soc = df_transform_3["Clay_SOC_ratio"].quantile(quantiles)
quantile_oc = df_transform_3["OC_g/kg"].quantile(quantiles)

# Checking the quantiles for 'Clay_SOC_ratio' and 'OC_g/kg
# print(f"Quantiles for Clay_SOC_ratio:\n{quantile_clay_soc} \n")
# print(f"Quantiles for OC_g/kg:\n{quantile_oc} \n", )


# calculate the quantile scores clay_soc_ratio
# clay_soc labels are reversed: more soc = good = lower clay_soc_ratio = excellent (label 4)
quantile_labels_clay_soc = [4, 3, 2, 1]
df_transform_3["Clay_SOC_score"] = pd.qcut(df_transform_3["Clay_SOC_ratio"], q=4, labels=quantile_labels_clay_soc)

# calculate quantile scores for oc
# labels ordered: more oc = good = higher value = excellent (label 4)
quantile_labels_oc = [1, 2, 3, 4]
df_transform_3["OC_score"] = pd.qcut(df_transform_3["OC_g/kg"], q=4, labels=quantile_labels_oc)

# change dtype to int (it is category otherwise)
df_transform_3["Clay_SOC_score"] = df_transform_3["Clay_SOC_score"].astype(int)
df_transform_3["OC_score"] = df_transform_3["OC_score"].astype(int)

# check
df_transform_3.head()

,POINT_ID,bulk_density_g/cm3,pH_CaCl2,EC_dS/m,OC_g/kg,Landcover,Landuse,erosion_by_water_t/ha/yr,P_kg/ha,N_kg/ha,K_kg/ha,Clay_SOC_ratio,Clay_SOC_score,OC_score
0,26762002,1.420,7.0,0.1241,8.6,Cropland,Agriculture (excluding fallow land and kitchen...,0.15,153.36,1988.0,580.78,8.14,3,1
1,27842416,0.955,3.7,0.1656,91.6,Woodland,Forestry,0.02,27.89,13179.0,116.89,1.42,4,4
2,26942016,1.089,3.9,0.0589,15.0,Woodland,Forestry,0.18,4.57,2613.6,83.20,8.67,3,2
3,26962018,1.088,5.0,0.1267,39.8,Woodland,Forestry,0.13,18.71,5222.4,239.58,4.52,3,4
4,30303360,0.943,4.8,0.1418,54.9,Grassland,Agriculture (excluding fallow land and kitchen...,0.00,75.82,9052.8,171.06,2.37,4,4


#### Predefined bins and score calculation for bulk density, EC, P, N, K and erosion
_The predefined bins are taken from the table of GSHI_Luis_Lado.pdf page 4(SHI Candidates)._

In [8]:
# Labels ascending: P, N, K: low = 1 (poor), high = 4 (excellent)
labels_asc = [1, 2, 3, 4]
bins_P = [0, 5, 10, 20, float('inf')]  # inf means "above 20"
bins_N = [0, 50, 100, 200, float('inf')]  # inf means "above 200"
bins_K = [0, 50, 100, 200, float('inf')]  # inf means "above 200"

# Labels descending: bulk density, EC, erosion: high = 1 (poor), low = 4 (excellent)
labels_desc = [4, 3, 2, 1]
bins_bulk_density = [0, 1.2, 1.4, 1.6, float('inf')] # inf means "above 1.6"
bins_EC = [0, 1.0, 2.0, 4.0, float('inf')] # inf means "above 4.0"
bins_erosion = [0, 1.0, 2.0, 5.0, float('inf')]


# calculating the scores_labels (scores) according to bins
df_transform_3["P_score"] = pd.cut(
    df_transform_3["P_kg/ha"], 
    bins=bins_P, 
    labels=labels_asc, 
    right=True,    # right=True means the right edge is INCLUDED
    include_lowest=True  # makes the first bin include 0
)

df_transform_3["N_score"] = pd.cut(
    df_transform_3["N_kg/ha"], 
    bins=bins_N, 
    labels=labels_asc, 
    right=True,    # right=True means the right edge is INCLUDED
    include_lowest=True  # makes the first bin include 0
)

df_transform_3["K_score"] = pd.cut(
    df_transform_3["K_kg/ha"], 
    bins=bins_K, 
    labels=labels_asc, 
    right=True,    # right=True means the right edge is INCLUDED
    include_lowest=True  # makes the first bin include 0
)

df_transform_3["bulk_density_score"] = pd.cut(
    df_transform_3["bulk_density_g/cm3"], 
    bins=bins_bulk_density, 
    labels=labels_desc, 
    right=True,    # right=True means the right edge is INCLUDED ', '
    include_lowest=True  # makes the first bin include 0
)

df_transform_3["EC_score"] = pd.cut(
    df_transform_3["EC_dS/m"], 
    bins=bins_EC, 
    labels=labels_desc, 
    right=True,    # right=True means the right edge is INCLUDED
    include_lowest=True  # makes the first bin include 0
)

df_transform_3["erosion_score"] = pd.cut(
    df_transform_3["erosion_by_water_t/ha/yr"], 
    bins=bins_erosion, 
    labels=labels_desc, 
    right=True,    # right=True means the right edge is INCLUDED
    include_lowest=True  # makes the first bin include 0
)

# change dtype to int (it is category otherwise)
df_transform_3["P_score"] = df_transform_3["P_score"].astype(int)
df_transform_3["N_score"] = df_transform_3["N_score"].astype(int)
df_transform_3["K_score"] = df_transform_3["K_score"].astype(int)
df_transform_3["bulk_density_score"] = df_transform_3["bulk_density_score"].astype(int)
df_transform_3["EC_score"] = df_transform_3["EC_score"].astype(int)
df_transform_3["erosion_score"] = df_transform_3["erosion_score"].astype(int)

# save
df_transform_3.to_csv("../output/df_transform_3.csv", index=False)

# check
df_transform_3.head()

,POINT_ID,bulk_density_g/cm3,pH_CaCl2,EC_dS/m,OC_g/kg,Landcover,Landuse,erosion_by_water_t/ha/yr,P_kg/ha,N_kg/ha,K_kg/ha,Clay_SOC_ratio,Clay_SOC_score,OC_score,P_score,N_score,K_score,bulk_density_score,EC_score,erosion_score
0,26762002,1.420,7.0,0.1241,8.6,Cropland,Agriculture (excluding fallow land and kitchen...,0.15,153.36,1988.0,580.78,8.14,3,1,4,4,4,2,4,4
1,27842416,0.955,3.7,0.1656,91.6,Woodland,Forestry,0.02,27.89,13179.0,116.89,1.42,4,4,4,4,3,4,4,4
2,26942016,1.089,3.9,0.0589,15.0,Woodland,Forestry,0.18,4.57,2613.6,83.20,8.67,3,2,1,4,2,4,4,4
3,26962018,1.088,5.0,0.1267,39.8,Woodland,Forestry,0.13,18.71,5222.4,239.58,4.52,3,4,3,4,4,4,4,4
4,30303360,0.943,4.8,0.1418,54.9,Grassland,Agriculture (excluding fallow land and kitchen...,0.00,75.82,9052.8,171.06,2.37,4,4,4,4,3,4,4,4


#### ph-CaCl score calculation - special predefined bins
(Transform #4)

_The nature of ph-values is that the middle values (around 7 neutral ph) are rather good, edge values (like 5 or 10) are rather poor for soil health. Bin assignment has to take this into consideration, we follow the suggestion of GSHI_Luis_Lado.pdf_

In [9]:
# for cell independence load df from csv
df_transform_4 = pd.read_csv("../output/df_transform_3.csv")

conditions = [
    (df_transform_4["pH_CaCl2"] < 5.0) | (df_transform_4["pH_CaCl2"] > 8.5),        # label 1: ph < 5.0, > 8.5
    ((df_transform_4["pH_CaCl2"] >= 5.0) & (df_transform_4["pH_CaCl2"] <= 6.0)) |   # label 2: ph 5.0 - 6.0 (includes 5.5-6.0), 7.5-8.5
    ((df_transform_4["pH_CaCl2"] >= 7.5) & (df_transform_4["pH_CaCl2"] <= 8.5)),
    ((df_transform_4["pH_CaCl2"] >= 6.0) & (df_transform_4["pH_CaCl2"] <= 6.5)) |   # label 3: ph 6.0 - 6.5, 7.0 - 7.5
    ((df_transform_4["pH_CaCl2"] >= 7.0) & (df_transform_4["pH_CaCl2"] <= 7.5)),
    ((df_transform_4["pH_CaCl2"] > 6.5) & (df_transform_4["pH_CaCl2"] < 7.0))       # label 4: ph 6.5 - 7.0
]

ph_labels = [1, 2, 3, 4]

df_transform_4["pH_score"] = np.select(conditions, ph_labels, default=np.nan)
df_transform_4["pH_score"] = df_transform_4["pH_score"].astype(int)

# save
df_transform_4.to_csv("../output/df_transform_4.csv", index=False)

# check
df_transform_4.head()

,POINT_ID,bulk_density_g/cm3,pH_CaCl2,EC_dS/m,OC_g/kg,Landcover,Landuse,erosion_by_water_t/ha/yr,P_kg/ha,N_kg/ha,...,Clay_SOC_ratio,Clay_SOC_score,OC_score,P_score,N_score,K_score,bulk_density_score,EC_score,erosion_score,pH_score
0,26762002,1.420,7.0,0.1241,8.6,Cropland,Agriculture (excluding fallow land and kitchen...,0.15,153.36,1988.0,...,8.14,3,1,4,4,4,2,4,4,3
1,27842416,0.955,3.7,0.1656,91.6,Woodland,Forestry,0.02,27.89,13179.0,...,1.42,4,4,4,4,3,4,4,4,1
2,26942016,1.089,3.9,0.0589,15.0,Woodland,Forestry,0.18,4.57,2613.6,...,8.67,3,2,1,4,2,4,4,4,1
3,26962018,1.088,5.0,0.1267,39.8,Woodland,Forestry,0.13,18.71,5222.4,...,4.52,3,4,3,4,4,4,4,4,2
4,30303360,0.943,4.8,0.1418,54.9,Grassland,Agriculture (excluding fallow land and kitchen...,0.00,75.82,9052.8,...,2.37,4,4,4,4,3,4,4,4,1


#### Removal of columns not needed anymore
(Transform #5)

_Aim is to keep only POINT_ID, the scores of each indicator and Landcover, Landuse - a clean dataset ready for SHI calculation._

In [10]:
# for cell independence load df from csv
df_transform_5 = pd.read_csv("../output/df_transform_4.csv")

# drop columns
columns_to_drop = ['Clay_SOC_ratio', 'OC_g/kg', 'P_kg/ha', 'N_kg/ha',
       'K_kg/ha', 'bulk_density_g/cm3', 'EC_dS/m', 'erosion_by_water_t/ha/yr',
       'pH_CaCl2']

df_transform_5.drop(columns=columns_to_drop, inplace=True)

# save
df_transform_5.to_csv("../output/df_transform_5.csv", index=False)

#
print(len(df_transform_5))

# check
df_transform_5.head()

4621


,POINT_ID,Landcover,Landuse,Clay_SOC_score,OC_score,P_score,N_score,K_score,bulk_density_score,EC_score,erosion_score,pH_score
0,26762002,Cropland,Agriculture (excluding fallow land and kitchen...,3,1,4,4,4,2,4,4,3
1,27842416,Woodland,Forestry,4,4,4,4,3,4,4,4,1
2,26942016,Woodland,Forestry,3,2,1,4,2,4,4,4,1
3,26962018,Woodland,Forestry,3,4,3,4,4,4,4,4,2
4,30303360,Grassland,Agriculture (excluding fallow land and kitchen...,4,4,4,4,3,4,4,4,1
